In [1]:
import numpy as np
import pandas as pd
import sklearn as sk
import yfinance as yf

In [2]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_predict,train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [4]:
model_quant=HistGradientBoostingRegressor(random_state=42)
model_cat=RandomForestRegressor(n_estimators=50,random_state=42)
model_embed = make_pipeline(StandardScaler(), SVR(kernel='rbf'))

Generate meta features!

In [8]:
df=pd.read_csv("../data/processed/cleaned_metrics.csv")
df.head()

,ticker,forwardPE,ev_to_ebitda,ebitda_margin,debt_to_equity,sector,industry,business_summary,ebitda,total_cash,total_debt,shares_outstanding
0,NVDA,17.693266,35.888,0.61698,7.255,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...,1.332300e+11,6.255600e+10,1.141200e+10,24300000000
1,GOOGL,24.995052,26.757,0.37279,16.133,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...,1.501750e+11,1.268430e+11,6.699600e+10,5822000000
2,AAPL,28.289606,25.736,0.35100,102.630,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ...",1.529020e+11,6.690700e+10,9.050900e+10,14681140000
3,MSFT,22.231121,17.616,0.57377,31.539,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...,1.752590e+11,8.946200e+10,1.232780e+11,7425629076
4,AMZN,26.568268,18.686,0.20327,43.435,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of...",1.457310e+11,1.230290e+11,1.785470e+11,10754251799


In [10]:
X=df[['forwardPE','ebitda_margin','debt_to_equity','ebitda','total_cash','total_debt','shares_outstanding','sector','industry','business_summary']]
Y=df['ev_to_ebitda']

In [11]:
from sklearn.model_selection import train_test_split

# 1. Split the data
# test_size=0.2 means 20% of data goes to testing, 80% to training
# random_state=42 ensures you get the same split every time you run the code
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [12]:
X_train.head()

,forwardPE,ebitda_margin,debt_to_equity,ebitda,total_cash,total_debt,shares_outstanding,sector,industry,business_summary
3,22.231121,0.57377,31.539,1.752590e+11,8.946200e+10,1.232780e+11,7425629076,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...
6,19.004490,0.50701,39.164,1.018920e+11,8.159200e+10,8.508100e+10,2187177748,Communication Services,Internet Content & Information,"Meta Platforms, Inc. engages in the developmen..."
27,19.734220,0.29047,68.719,2.476500e+10,1.082500e+10,3.663900e+10,2324000685,Consumer Defensive,Household & Personal Products,The Procter & Gamble Company provides branded ...
31,76.650480,0.32179,3.063,1.440160e+09,7.177043e+09,2.293380e+08,2291470751,Technology,Software - Infrastructure,Palantir Technologies Inc. builds and deploys ...
19,43.940975,0.04702,60.257,1.346000e+10,1.112600e+10,1.022800e+10,443652540,Consumer Defensive,Discount Stores,"Costco Wholesale Corporation, together with it..."


In [13]:
quant_columns = [
    'forwardPE','ebitda_margin','debt_to_equity','ebitda','total_cash','total_debt','shares_outstanding'
]

In [14]:
X_quant_train = X_train[quant_columns]
X_quant_test = X_test[quant_columns]


In [15]:
scaler = StandardScaler()
X_quant_train = scaler.fit_transform(X_quant_train)

In [34]:
embed_columns = [col for col in X_train.columns if col.startswith('dim_')]
X_embed_train = X_train[embed_columns]
X_embed_test = X_test[embed_columns]

In [35]:
cat_key_columns=["industryKey","sector","country"]
cat_columns = [
    col for col in X_train.columns 
    if any(keyword in col for keyword in cat_key_columns)
]
X_cat_train=X_train[cat_columns]
X_cat_test=X_test[cat_columns]

In [36]:
Y_train

,symbol,enterpriseToEbitda
0,LOW,13.959
1,SRE,20.033
2,ALGN,13.132
3,AME,22.090
4,TYL,32.374
...,...,...
308,RMD,15.840
309,HD,15.388
310,ADBE,10.294
311,MA,21.856


In [37]:
Y_train_series = Y_train['enterpriseToEbitda']
Y_train_series

0      13.959
1      20.033
2      13.132
3      22.090
4      32.374
        ...  
308    15.840
309    15.388
310    10.294
311    21.856
312    34.148
Name: enterpriseToEbitda, Length: 313, dtype: float64

In [38]:
pred_quant = cross_val_predict(model_quant, X_quant_train, Y_train_series, cv=5)
pred_embed = cross_val_predict(model_embed, X_embed_train, Y_train_series, cv=5)
pred_cat=cross_val_predict(model_cat,X_cat_train,Y_train_series,cv=5)

In [39]:
X_meta_train = np.column_stack((pred_quant, pred_cat,pred_embed))

In [40]:
model_quant.fit(X_quant_train, Y_train_series)
model_embed.fit(X_embed_train, Y_train_series)
model_cat.fit(X_cat_train,Y_train_series)
pred_quant_test = model_quant.predict(X_quant_test)
pred_embed_test = model_embed.predict(X_embed_test)
pred_cat_test=model_cat.predict(X_cat_test)
X_meta_test = np.column_stack((pred_quant_test, pred_cat_test,pred_embed_test))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but HistGradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [41]:
from sklearn.linear_model import Lasso

In [42]:
judge_model = Lasso(alpha=0.5, positive=True)
judge_model.fit(X_meta_train,Y_train_series)
y_final_pred=judge_model.predict(X_meta_test)

In [43]:
from sklearn.metrics import r2_score, mean_squared_error

In [44]:
Y_test_series=Y_test["enterpriseToEbitda"]

In [45]:
test_r2=r2_score(Y_test_series,y_final_pred)
test_mse=mean_squared_error(Y_test_series,y_final_pred)

In [46]:
test_r2

-5.3482627462986505

In [47]:
test_mse

749.8771336079059

In [48]:
quant_r2 = r2_score(Y_train_series, X_meta_train[:, 0])
cat_r2 = r2_score(Y_train_series, X_meta_train[:, 1])
embed_r2 = r2_score(Y_train_series, X_meta_train[:, 2])

print("--- Training Set R2 for Each Expert ---")
print(f"Quant Expert R2: {quant_r2:.4f}")
print(f"Cat Expert R2:   {cat_r2:.4f}")
print(f"Embed Expert R2: {embed_r2:.4f}")

--- Training Set R2 for Each Expert ---
Quant Expert R2: -0.1003
Cat Expert R2:   -0.1408
Embed Expert R2: -0.0037
